# Übungen Woche 9: Sichtbarkeit & Information Hiding

In diesem Übungsnotebook vertiefen Sie die Zugriffsmodifikatoren in Java (`public`, `protected`, *package-private*, `private`) sowie das Prinzip des **Information Hiding** (Datenkapselung).

---
## Teil 2: Sichtbarkeitsattribute setzen

Setzen Sie in den folgenden Programmen über mehrere Klassen und Packages hinweg für **jede Klasse, jedes Feld, jeden Konstruktor und jede Methode** das **strengstmögliche** Sichtbarkeitsattribut (`private` < *package-private* < `protected` < `public`), sodass sich alles noch fehlerfrei kompilieren lässt.

### Aufgabe 2.1: Klassen und Methoden über zwei Packages

*Hinweis: Leider ist Sichtbarkeit in Jupyter-Notebooks etwas instabil. Es kann sein, dass eine Zelle mehrere Male kompiliert werden muss, bis ein Sichtbarkeitsfehler auftritt. Bei gegenseitigen Package-Abhängigkeiten (`A` und `B`) müssen beim allerersten Kompilieren eventuell die Aufrufe auf das jeweils andere Package kurz auskommentiert und anschliessend wieder einkommentiert werden. Wenn ein Fehler behoben wird (z. B. durch Lockern der Sichtbarkeit), sollten Sie ebenfalls sicherstellen, dass beide Zellen neu kompiliert werden. (Führen Sie die Methoden hier nicht aus, da sie eine Endlosrekursion enthalten – es geht rein um die Kompilierbarkeit.)*

In [ ]:
package A;

import B.B1;

public class A1 {

    public static void a11() {
        System.out.println("a11");
        A1.a12();
    }

    public static void a12() {
        System.out.println("a12");
        A2.a21();
    }
}

public class A2 {

    public static void a21() {
        System.out.println("a21");
        A1.a11();
    }

    public static void a22() {
        System.out.println("a22");
        B1.b12();
    }
}

In [ ]:
package B;

import A.A1;
import A.A2;

public class B1 extends A1 {

    public static void b11() {
        System.out.println("b11");
        A1.a11();
    }

    public static void b12() {
        System.out.println("b12");
        A2.a22();
        B1.b11();
    }
}

### Aufgabe 2.2: Klassen, Felder, Konstruktoren und Methoden über zwei Packages

Schränken Sie auch in diesem Programm alle Sichtbarkeitsmodifikatoren so weit wie möglich ein, sodass `ShopApp.main(...)` weiterhin problemlos kompiliert und ausgeführt werden kann.

In [ ]:
package verlag;

public class Buch {
    public String titel;
    public String interneNotiz;
    public double basisPreis;

    public Buch(String titel, double basisPreis, String interneNotiz) {
        this.titel = titel;
        this.basisPreis = basisPreis;
        this.interneNotiz = interneNotiz;
    }

    public String getInterneNotiz() {
        return this.interneNotiz;
    }

    public String formatiereInfo() {
        return this.titel + " (" + this.getInterneNotiz() + ")";
    }
}

public class VerlagsKatalog {
    public static void druckeKatalogEintrag(Buch buch) {
        System.out.println(buch.titel + ": " + buch.formatiereInfo());
    }
}

In [ ]:
package shop;

import verlag.Buch;
import verlag.VerlagsKatalog;

public class SonderEdition extends Buch {
    public double rabatt;

    public SonderEdition(String titel, double basisPreis, double rabatt) {
        super(titel, basisPreis, "Limitierte Auflage");
        this.rabatt = rabatt;
    }

    public double berechneEndpreis() {
        return this.basisPreis * (1.0 - this.rabatt);
    }
}

public class ShopApp {
    public static void main(String[] args) {
        SonderEdition edition = new SonderEdition("Java Kompakt", 40.0, 0.25);
        VerlagsKatalog.druckeKatalogEintrag(edition);
        System.out.println("Endpreis: " + edition.berechneEndpreis() + " CHF");
    }
}

ShopApp.main(new String[0]);

---
## Teil 3: Information Hiding

Jede der folgenden Aufgaben besteht aus zwei Code-Zellen:
1. Die **erste Zelle** enthält ein kleines Programm mit der regulären `Main`-Klasse.
2. Die **zweite Zelle** enthält eine «böse» Klasse (`EagerReader` bzw. `Villain`), die das Programm sabotiert oder Schutzmechanismen umgeht.

**Ihre Aufgabe:** Schreiben Sie jeweils das Programm in der **ersten Zelle** so um (z. B. mit geeigneten Zugriffsmodifikatoren und ggf. neuen Zugriffsmethoden), dass:
- die reguläre `Main`-Klasse weiterhin fehlerfrei kompiliert und die gleiche Ausgabe erzeugt,
- die böse Klasse in der **zweiten Zelle nicht mehr kompiliert**.

**Wichtig:** Die böse Klasse in der zweiten Zelle darf dabei **nicht verändert** werden!

### Aufgabe 3.1: Geschütztes Buch (`ProtectedBook`)

In [ ]:
public class ProtectedBook {

    String title;

    public ProtectedBook(String title) {
        this.title = title;
        if (title != null) {
            System.out.println("Created the following Book: " + title);
        }
    }

    private static ProtectedBook createEmptyBook() {
        return new ProtectedBook(null);
    }

    public static ProtectedBook getBook(String title, String password) {
        if (password == "reader") {
            return new ProtectedBook(title);
        }
        else {
            System.out.println("Wrong password! Returned empty book.");
            return createEmptyBook();
        }
    }

}

public class Main {

    public static void main(String[] args) {
        ProtectedBook.getBook("Lord of the Rings", "readr");
        System.out.println("Oops, mistyped");
        ProtectedBook.getBook("Lord of the Rings", "reader");
    }
}

Main.main(new String[0]);

In [ ]:
public class EagerReader {

    public static void main(String[] args) {
        ProtectedBook.getBook("Lord of the Rings", "?");
        System.out.println("I don't know the password");
        new ProtectedBook("Illegal Lord of the Rings");
    }
}

EagerReader.main(new String[0]);

### Aufgabe 3.2: Spieleinstellungen (`GameSettings`)

In [ ]:
public class GameSettings {
    public static int maxAttempts = 3;

    public static void display() {
        System.out.println(
            "Maximum attempts: " + maxAttempts
        );
    }
}

public class Main {

    public static void playGame() {
        for (int i = 0; i < GameSettings.maxAttempts; i++) {
            System.out.println("Playing...");
        }
    }

    public static void main(String[] args) {
        GameSettings.display();
        playGame();
    }
}

Main.main(new String[0]);

In [ ]:
public class Villain {

    public static void sabotage() {
        GameSettings.maxAttempts = 10;
        System.out.println("I want to play longer!");
    }

    public static void main(String[] args) {
        GameSettings.display();
        sabotage();
        GameSettings.display();

        Main.playGame();
    }
}

Villain.main(new String[0]);

### Aufgabe 3.3: Lagerbestand (`Inventory`)

In [ ]:
public class Inventory {
    private int[] stock;

    public Inventory() {
        stock = new int[]{10, 20, 30};
    }

    public int[] getStock() {
        return stock;
    }

    public int stockSize() {
        return stock.length;
    }

    public void display() {
        for (int amount : stock) {
            System.out.println(amount);
        }
    }
}

public class Main {

    public static int totalAmount(Inventory inventory) {
        int[] stock = inventory.getStock();
        int n = inventory.stockSize();
        int total = 0;
        for (int i = 0; i < n; i++) {
            total += stock[i];
        }
        return total;
    }

    public static void main(String[] args) {
        Inventory inventory = new Inventory();

        inventory.display();
        System.out.println("Total:");
        System.out.println(totalAmount(inventory));
    }
}

Main.main(new String[0]);

In [ ]:
public class Villain {

    public static void sabotage(Inventory inventory) {
        int[] stock = inventory.getStock();

        stock[0] = 0;
        stock[1] = 0;
        stock[2] = 0;

        System.out.println("Muahahaha!");
    }

    public static void main(String[] args) {
        Inventory inventory = new Inventory();

        inventory.display();
        sabotage(inventory);
        inventory.display();
    }
}

Villain.main(new String[0]);

### Aufgabe 3.4: Zähler (`Counter`)

In [ ]:
public class Counter {
    public int count;

    public Counter() {
        this.count = 0;
    }

    public void display() {
        System.out.println(this.count);
    }

}

public class Main {

    public static void countEven(int[] arr, Counter counter) {
        for (int i : arr) {
            if (i % 2 == 0) {
                counter.count++;
            }
        }
    }

    public static void main(String[] args) {
        Counter evenCounter = new Counter();
        int[] arr = new int[]{1,2,5,8,3,6};
        countEven(arr, evenCounter);
        evenCounter.display();

    }
}

Main.main(new String[0]);

In [ ]:
public class Villain {

    public static void stealCount(Counter counter) {
        counter.count = 0;
        System.out.println("Muahahaha!");
    }

    public static void main(String[] args) {
        Counter evenCounter = new Counter();
        int[] arr = new int[]{1,2,5,8,3,6};
        Main.countEven(arr, evenCounter);
        evenCounter.display();
        stealCount(evenCounter);
        evenCounter.display();

    }

}

Villain.main(new String[0]);